# Roulette Examination

Implementation of a soft-selection algorithm. 

The matrices reach very high alignment scores, but they are not all stable and their fitness is not quite optimal. In fact, the average for noisy scenarios is that they are not stable!

In [ ]:
# Packages
display("text/html", "<style>.container { width:100% !important; }</style>")

# Julia dependencies for plotting, stats, and experiments
# using Plots, LaTeXStrings, Distributions, DoubleFloats, Plots.PlotMeasures, LinearAlgebra,
    DataFrames, CSV, LsqFit, StatsBase, Distances, Parameters, Random, StatsBase, Graphs, 
    GraphPlot, Base.Threads, Printf, JLD2

# Local modules
include("../src/wagner_algorithm.jl")
using ..BooleanNetwork: SimulationParameters, SimulationData
using ..BooleanNetwork: run_simulation
include("../src/data_processing.jl")
using ..CustomStats: summarize_history, summarize_simulation_run, alignment_score, compute_mut_robustness
include("../scripts/run_experiments.jl")
using ..ExperimentHandler: Experiment1Config, Experiment2Config, Experiment1Result, Experiment2Result, MotifStats
using ..ExperimentHandler: run_experiment_1, run_experiment_2, build_noise_scenarios

# Utilities
using ProgressMeter
using Base.Threads: Atomic, atomic_add!

# Python bridge for motif search utilities
using PyCall
py"""
import sys
import os
sys.path.append(os.path.abspath("../src/"))
from motif_search import *
"""

In [ ]:
# Plotting utilities
const ROOT_DIR = normpath(joinpath(pwd(), ".."))
const FIGURES_DIR = joinpath(ROOT_DIR, "figures")
mkpath(FIGURES_DIR)
const ROBUSTNESS_FIG_DIR = joinpath(FIGURES_DIR, "scatterplots_mutational_robustness")
mkpath(ROBUSTNESS_FIG_DIR)

default(dpi=900, legendfontsize=24, guidefont=font(24), tickfont=font(24), margin=8Plots.mm, grid=false)
const BONFERRONI_TESTS = 2

"""
    save_plot(plot_obj, filename)

Save a figure to the figures directory and print the output path.
"""
function save_plot(plot_obj, filename)
    filepath = joinpath(FIGURES_DIR, filename)
    savefig(plot_obj, filepath)
    println("Saved $(filepath)")
end

In [ ]:
Plots.theme(:dao)
function compare_histograms(
    data_vecs, labels;
    bins=30, xlabel="", ylabel="", title="",c_pal=:blues, normalize=:pdf)

    p1 = plot()

    colors = palette(c_pal, size(data_vecs, 1))

    for (idx, data_vec) in enumerate(collect(eachrow(data_vecs)))
        histogram!(p1, data_vec,
        bins=bins,
        xlabel=xlabel,
        ylabel=ylabel,
        label=labels[idx],
        title=title,
        color=colors[idx],
        normalize=normalize,
        alpha=0.30
        )
    end

    return p1   
end

In [ ]:
simulation_parameters = SimulationParameters(generations=1000, selection_type=:roulette)
noiseless_results = run_simulation(simulation_parameters) # evolved without noise

In [ ]:
noiseless_final_fit_avg = mean(noiseless_results.fitness_history[end, :])
println("Average fitness at generation 1000 is $noiseless_final_fit_avg")

p11 = compare_histograms(noiseless_results.fitness_history[[1, 300, end], :],
    ["Generation 1", "Generation 300", "Generation 1000"];
    bins=0.0:0.1:1.1, xlabel="Fitness", ylabel="Frequency", 
    title="Fitness in Noiseless Scenario", c_pal=:blues, normalize=false
)
vline!(p11, [noiseless_final_fit_avg], color=:blue,
    label=L" $\mu_{1000} = $" * "$(round(noiseless_final_fit_avg, digits = 3))")
display(p11)

It does seem that most of the matrices tend to converge to a very high fitness. However, the smaller values given by admitting unstable mutations and recombinations in the next generation skew the distribution to the right. We also examine the completion

In [ ]:
p12 = compare_histograms(noiseless_results.completion_history[[1, 300, end], :],
    ["Generation 1", "Generation 300", "Generation 1000"];
    bins=0.0:0.1:1.1, xlabel="Is Stable?", ylabel="Frequency",
    title="Completion in Noiseless Scenario", c_pal=:Purples, normalize=false
)

In [ ]:
alignment_noiseless = CustomStats.alignment_score.(noiseless_results.matrices_history, Ref(noiseless_results.optimal_phenotype))
noiseless_final_alig_avg = mean(alignment_noiseless[end, :])
println("Average alignemnt at generation 1000 is $noiseless_final_alig_avg")

p13 = compare_histograms(alignment_noiseless[[1, 300, end], :],
    ["Generation 1", "Generation 300", "Generation 1000"];
    bins=-1.0:0.05:1.1, xlabel="Alignment Score", ylabel="Frequency",
    title="Alignment in Noiseless Scenario", c_pal=:reds, normalize=false
)

vline!(p13, [noiseless_final_alig_avg], color=:red,
    label=L" $\mu_{1000} = $" * "$(round(noiseless_final_alig_avg, digits = 3))")
display(p13)

In [ ]:
p1 = plot(p11, p12, p13; layout=(1, 3), size=(1500, 400), legend=:best)
save_plot(p1, "roulette_analysis/single_run_noiseless.png")
display(p1)

In [ ]:
simulation_parameters = SimulationParameters(generations=1000, selection_type=:roulette,
    noise_dist=Gamma(1.0,1.0))
noisy_results = run_simulation(simulation_parameters);

In [ ]:
noisy_final_fit_avg = mean(noisy_results.fitness_history[end, :])
println("Average fitness at generation 1000 is $noisy_final_fit_avg")

p21 = compare_histograms(noisy_results.fitness_history[[1, 300, end], :],
    ["Generation 1", "Generation 300", "Generation 1000"];
    bins=0.0:0.1:1.1, xlabel="Fitness", ylabel="Frequency",
    title=L"Fitness in Moderate Noise $\theta=1.0$ Scenario", c_pal=:blues,
    normalize=false)
vline!(p21, [noisy_final_fit_avg], color=:blue, 
label=L" $\mu_{1000} = $" * "$(round(noisy_final_fit_avg, digits = 3))")
display(p21)

For $\theta = 1.0$ (i.e., noise_dist = $\Gamma(1.0,1.0)$), it seems still reasonably similar to the noiseless scenario. We see a small increase in the number of very low fitness values, presumably due to completeness. Let's check that.

In [ ]:
p22 = compare_histograms(noisy_results.completion_history[[1, 300, end], :],
    ["Generation 1", "Generation 300", "Generation 1000"];
    bins=0.0:0.1:1.1, xlabel="Is Stable?", ylabel="Frequency",
    title=L"Completion in Moderate Noise $\theta=1.0$ Scenario", c_pal=:Purples,
    normalize=false
)

In [ ]:
alignment_noisy = CustomStats.alignment_score.(noisy_results.matrices_history, Ref(noisy_results.optimal_phenotype))
noisy_final_alig_avg = mean(alignment_noisy[end, :])
println("Average alignemnt at generation 1000 is $noisy_final_alig_avg")

p23 = compare_histograms(alignment_noisy[[1, 300, end], :],
    ["Generation 1", "Generation 300", "Generation 1000"];
    bins=-1.0:0.05:1.1, xlabel="Alignment Score", ylabel="Frequency",
    title=L"Alignment in Noisy $\theta=1.0$ Scenario", c_pal=:reds, normalize=false
)

vline!(p23, [noisy_final_alig_avg], color=:red,
    label=L" $\mu_{1000} = $" * "$(round(noisy_final_alig_avg, digits = 3))")
display(p23)

In [ ]:
p2 = plot(p21, p22, p23; layout=(1, 3), size=(1500, 400), legend=:best)
save_plot(p2, "roulette_analysis/single_run_noisy.png")
display(p2)

In the noiseless scenario, about 10 matrices resulted to be unstable in the final generation, whereas more than 50 are unstable in the noisy scenario. Let's examine what happens in the very noisy scenario.

In [ ]:
simulation_parameters = SimulationParameters(generations=1000, selection_type=:roulette,
    noise_dist=Gamma(1.0/8.0, 8.0))
very_noisy_results = run_simulation(simulation_parameters);

In [ ]:
very_noisy_final_fit_avg = mean(very_noisy_results.fitness_history[end, :])
println("Average fitness at generation 1000 is $very_noisy_final_fit_avg")

p31 = compare_histograms(very_noisy_results.fitness_history[[1, 300, end], :],
    ["Generation 1", "Generation 300", "Generation 1000"];
    bins=0.0:0.1:1.1, xlabel="Fitness", ylabel="Frequency",
    title=L"Fitness in Very Noisy $\theta=8.0$ Scenario", c_pal=:blues,
    normalize=false)
vline!(p31, [very_noisy_final_fit_avg], color=:blue, 
label=L" $\mu_{1000} = $" * "$(round(very_noisy_final_fit_avg, digits = 3))")
display(p31)

In [ ]:
p32 = compare_histograms(very_noisy_results.completion_history[[1, 300, end], :],
    ["Generation 1", "Generation 300", "Generation 1000"];
    bins=0.0:0.1:1.1, xlabel="Is Stable?", ylabel="Frequency",
    title=L"Completion in Very Noisy $\theta=8.0$ Scenario", c_pal=:Purples,
    normalize=false
)

In [ ]:
alignment_very_noisy = CustomStats.alignment_score.(very_noisy_results.matrices_history, Ref(very_noisy_results.optimal_phenotype))
very_noisy_final_alig_avg = mean(alignment_very_noisy[end, :])
println("Average alignemnt at generation 1000 is $very_noisy_final_alig_avg")

p33 = compare_histograms(alignment_very_noisy[[1, 300, end], :],
    ["Generation 1", "Generation 300", "Generation 1000"];
    bins=-1.0:0.05:1.1, xlabel="Alignment Score", ylabel="Frequency",
    title=L"Alignment in Very Noisy $\theta=1.0$ Scenario", c_pal=:reds, normalize=false
)

vline!(p33, [very_noisy_final_alig_avg], color=:red,
    label=L" $\mu_{1000} = $" * "$(round(very_noisy_final_alig_avg, digits = 3))")
display(p33)

In [ ]:
p3 = plot(p31, p32, p33; layout=(1, 3), size=(1500, 400), legend=:best)
save_plot(p3, "roulette_analysis/single_run_very_noisy.png")
display(p3)

About 125 matrices are unstable in the very noisy scenario, but the alignment score is a quantity that is conserved, and is perhaps a stronger evolutionary marker than fitness alone. We will make an analysis about mutational robustness and alignment score separately. The roulette selection has no filter to not choose unstable matrices as a result of mutation, recombination, and noise, which leads to many generations with too many unstable matrices. In particular, changes in phenotype are minimized by maximizing alignment, but noise is too strong!